# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [4]:
import os
import gradio as gr
import google.generativeai
import anthropic
from openai import OpenAI
from dotenv import load_dotenv
#from IPython.display import Markdown, display, update_display

In [5]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if deepseek_api_key:
    print(f"Deepseek API Key exists and begins {deepseek_api_key[:8]}")
else:
    print("Deepseek API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyBv
Deepseek API Key exists and begins sk-e18f8


In [6]:
system_message = "You are a helpful programming assistant called Coding-Ally."
system_message += "Give descriptive, detailed and easy to understand explanations to the programming questions/problems posed to you"
system_message += "Always be accurate. If you don't know the answer, say so."

In [7]:
openai = OpenAI()
claude = anthropic.Anthropic()
google.generativeai.configure()
deepseek_via_openai_client = OpenAI(
    api_key=deepseek_api_key, 
    base_url="https://api.deepseek.com"
)

In [8]:
def chat_openai_mini(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, stream=True)

    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [9]:
def chat_openai_4o(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o", messages=messages, stream=True)

    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [18]:
def chat_claude(message, history):
    messages = history + [{"role": "user", "content": message}]
    response = claude.messages.stream(
        model="claude-3-7-sonnet-latest",
        max_tokens=500,
        temperature=0.7,
        system=system_message,
        messages=messages
    )

    result = ""
    with response as stream:
        for text in stream.text_stream:
            result += text or ""
            yield result

In [11]:
def chat_deepseek(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = deepseek_via_openai_client.chat.completions.create(model="deepseek-chat", messages=messages, stream=True)

    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [12]:
def code_ally(message, history, model):
    
    if model=="gpt-4o-mini":
        yield from chat_openai_mini(message, history)
    elif model=="gpt-4o":
       yield from chat_openai_4o(message, history)
    elif model=="claude-sonnet":
        yield from chat_claude(message, history)
    elif model=="deepseek":
        yield from chat_deepseek(message, history)
    else:
        raise ValueError("Unknown model")


In [19]:

with gr.Blocks() as demo:

    gr.Markdown("## 🤖 I'm CodeAlly, your coding assistant. How can I help you today?")

    model_selector = gr.Dropdown(
        choices=["gpt-4o-mini", "gpt-4o", "claude-sonnet", "deepseek"],
        label="Choose a Model",
        value="gpt-4o-mini"
    )

    chat = gr.ChatInterface(
        fn=code_ally,
        additional_inputs=[model_selector],
        type="messages"
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.
